In [2]:
%pip install sympy casadi matplotlib scipy -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sympy
import scipy.integrate
import matplotlib.pyplot as plt
import numpy as np

We like Object Orientedness and defining class etc. because it keeps things clean.

In [ ]:
class RoverModel:

    def __init__(self):
        # whatever we need to know to define the system goes here

        self.t = sympy.symbols("t") # time
        self.delta = sympy.symbols("Delta")    
    

        # state
        x = sympy.symbols("x")
        y = sympy.symbols("y")
        theta = sympy.symbols("theta")
        self.x = sympy.Matrix([x, y, theta])

        # input
        u = sympy.symbols("u")
        omega = sympy.symbols("omega")
        self.u = sympy.Matrix([u, omega])

        # noise 
        # process noise

        w_v = sympy.symbols("w_v") # No Slide Slip
        w_omega = sympy.symbols("w_omega")
        self.w = sympy.Matrix([w_v, w_omega])
        

        # measurement noise (just GPS)
        v_y = sympy.symbols("v_y")
        v_x = sympy.symbols("v_x")
        self.v = sympy.Matrix([v_x, v_y]) 


    def dynamics(self):
        
        theta = self.x[2]
        v = self.u[0]
        omega = self.u[1]
        w_v = self.w[0]
        w_omega = self.w[1]
        cos = sympy.cos
        sin = sympy.sin
        x_dot = sympy.Matrix([
            (v + w_v) * cos(theta),
            (v + w_v) * sin(theta),
            (omega + w_omega)
        ])
        return x_dot
    
    def gps_measurement(self):
        # measurement
        x = self.x[0]
        y = self.x[1]
        v_x = self.v[0]
        v_y = self.v[1]
        return sympy.Matrix([x + v_x, y + v_y])

model =  RoverModel()

In [ ]:
model.x

In [ ]:
model.dynamics()

In [ ]:
model.gps_measurement()

Continuous function with a KF is stupid to work with, we discretize our dynamics and measurement eqns

In [ ]:
A = model.dynamics().jacobian(model.x)
A

In [ ]:
B = model.dynamics().jacobian(model.u)
B

In [ ]:
Ad = (A * model.delta).exp()
Ad

In [ ]:
exp_int = sympy.integrate((A * model.t).exp(), (model.t, 0, model.delta))
exp_int

In [ ]:
Bd = exp_int  @ B
Bd

Discretizing gps measurements

In [ ]:
H = model.gps_measurement().jacobian(model.x)
H

We are taking a snapshot of the Jacobian and that's the only way we can solve by linearizing. Theta here is 0. 

The Std KF does not realize like Lie Grp Theroy that these theta's (and other state variables) are continuously changing

In [ ]:
model.dynamics()

In [ ]:
f_eval = sympy.lambdify(
    [model.x, model.u, model.w],
    model.dynamics().T.tolist()[0]
)

Lambidfy converts Py code to C code in back end and hence, making it faster for computations

In [ ]:
f_eval([1, 1, 1], [1, 1], [0, 0])

In [ ]:
t0 = 0
tf = 10
t_eval = np.linspace(t0, tf, 1000)
res = scipy.integrate.solve_ivp(
    fun = lambda t, y: f_eval(y, [1, 1], [0, 0]), 
    t_span = [t0, tf], 
    y0 = [0, 0, 0], 
    t_eval = t_eval
    )

In [ ]:
plt.plot(res.t, res.y.T)

In [ ]:
plt.plot(res.y[0], res.y[1])

Theta's winding up, because you have an estimate of Jacobian, and ODE45 (RK45 here) is trying to integrate and error keeps winding up for theta, as seen in Green line in 1st plot. 

You are better off using Matrix Exponential since they give you exact theta values. 